Optimisation de l'entrainement pour `focus` - à matche avec 10..

> ... TODO ...


In [1]:
from retinotopy import *

HOST='obiwan.local'
Running on metal mps
Welcome on macOS-14.4.1-arm64-arm-64bit
On date 2024-04-25, Running learning on host obiwan.local with device mps


In [2]:
data_set_type = 'focus' # Select your root between : 'boxes', 'focus', 'full'
print(f'{data_set_type=}')
args = Params()
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training

data_set_type='focus'


# Training function

In [3]:
def train_model(args, model, dataloaders, each_steps=64, verbose=True):
    # if torch.cuda.is_available():
    #     model = model.to(device, memory_format=torch.channels_last)
    #     torch.cuda.amp.GradScaler(enabled=True)
    # else:
    #     model = model.to(device)
    model = model.to(device)
    # retraining the full model
    for param in model.parameters():
        param.requires_grad = True        

    if args.beta2 > 0.: 
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(1-args.momentum, 1-args.beta2)) 
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=1-args.momentum) # to set training variables

    df_train = pd.DataFrame([], columns=['epoch', 'i_image', 'total_image', 'avg_loss', 'avg_acc', 'avg_loss_val', 'avg_acc_val', 'time']) 

    criterion = nn.CrossEntropyLoss() #binary_cross_entropy_with_logits
    total_image = 0
    since = time.time()

    n_train = len(dataloaders['train'].dataset)
    n_train_stop = args.n_train_stop
    if n_train_stop==0: n_train_stop = n_train

    for i_epoch in range(args.num_epochs):
        i_image = 0
        for i_step, (images, labels) in enumerate(dataloaders['train']):
            images, labels = images.to(device), labels.to(device)
            total_image += len(images)
            i_image += len(images)
            if i_image > n_train_stop: break

            optimizer.zero_grad()

            outputs = model(images)
             
            loss = criterion(outputs, labels)            
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs.data, dim=1)

            avg_loss = loss.item() * images.size(0)
            avg_acc = torch.mean((preds == labels.data)*1.).cpu().item()

            if (i_step % (max(n_train_stop//args.batch_size//each_steps, 1))==0) or (i_step == n_train_stop-1):
                with torch.no_grad():
                    loss_val = 0
                    acc_val = 0
                    model = model.eval()
                    n_val = len(dataloaders['val'])
                    for _, (images, labels) in enumerate(dataloaders['val']):
                        images, labels = images.to(device), labels.to(device)

                        outputs = model(images)

                        loss = criterion(outputs, labels)

                        loss_val += loss.item() * images.size(0)

                        _, preds = torch.max(outputs.data, dim=1)
                        acc_val += torch.mean((preds == labels.data)*1.).cpu().item()

                    avg_loss_val = loss_val / n_val
                    avg_acc_val = acc_val / n_val

                    df_train.loc[len(df_train)] = {'epoch': i_epoch, 'i_image':i_image, 'total_image':total_image, 'avg_loss':avg_loss, 'avg_acc':avg_acc, 'avg_loss_val':avg_loss_val, 'avg_acc_val':avg_acc_val, 'time':time.time() - since}
                    if verbose:  print(f"Epoch {i_epoch}, i_image {i_image} : train= loss: {avg_loss:.4f} / acc : {avg_acc:.4f} - val= loss : {avg_loss_val:.4f} / acc : {avg_acc_val:.4f} / time:{time.time() - since:.1f}")

    if torch.cuda.is_available(): torch.cuda.empty_cache()        
    return model, df_train

# optimize meta-parameters

In [4]:
# print(path_save)
# %ls -l {path}*
# %rm {path} + '.sqlite3'

In [5]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [6]:
study_name = datetag + '_optuna'

model_name = 'resnet101'
do_polar = True

model_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}.pt'


scan_dicts= {
             'image_size' : [224],
            }

label_dicts= {
             'image_size' : 'image size',
            }

In [7]:
model_filename

'cached_data/2024-04-25_focus_resnet101_do_polar=True.pt'

In [8]:
%ls {model_filename}

ls: cached_data/2024-04-25_focus_resnet101_do_polar=True.pt: No such file or directory


In [9]:
subplotpars_scan = SubplotParams(left=0.125, right=.95, bottom=0.25, top=.975)
max_threshold = .999
for key in scan_dicts:
    filename = f'{data_cache}/{study_name}_{key}.json'
    if not(os.path.isfile(filename)):
        print(50*'=')
        print('Scanning along', key, "=", label_dicts[key])
        print(50*'=')
        if os.path.isfile(filename):
            df_scan = pd.read_json(filename)
        else:
            measure_columns = [key, 'accuracy']
            df_scan = pd.DataFrame([], columns=measure_columns)
            i_loc = 0
            for i_value, value in enumerate(scan_dicts[key]):
                print('i_value', i_value + 1, ' /', len(scan_dicts[key]), key, '=', value)

                opt =  Params()
                opt.root = f'{DATAROOT}/Imagenet_boxes' # Directory containing images to perform the training
                opt.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training
                opt.n_train_stop = 100000
                opt.num_epochs = 1
                # opt.beta2 = 1.e-7

                new_dict = asdict(opt)
                new_dict[key] = value
                new_opt = Params(**new_dict)
                
                def objective(trial):
                    new_opt.rs_min = trial.suggest_float('rs_min', -1, 1.)
                    new_opt.rs_max = trial.suggest_float('rs_max', -6, -4)
                    scale = 4
                    new_opt.momentum = trial.suggest_float('momentum', opt.momentum/scale, min(opt.momentum*scale, max_threshold), log=True)
                    scale = 10
                    new_opt.lr = trial.suggest_float('lr', opt.lr / scale, opt.lr * scale, log=True)
                    scale = 50
                    # new_opt.beta2 = trial.suggest_float('beta2', opt.beta2/scale, min(opt.beta2*scale, 1.), log=True)

                    # load legacy model
                    if model_name == 'resnet50':
                        model_retrain = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
                    else:
                        model_retrain = torchvision.models.resnet101(weights=torchvision.models.ResNet101_Weights.DEFAULT)
                                        
                    # modify padding in resnet to make it circular
                    model_retrain = make_padding_circular_again(model_retrain)
                    model_retrain = model_retrain.to(device)

                    # use warmstart from center model
                    model_retrain.load_state_dict(torch.load(model_filename, map_location=torch.device(device)))
                
                    # train and get accuracy on the validation set
                    dataloaders = datasets_transforms(new_opt, verbose=False)
                    _, df_train = train_model(new_opt, model_retrain, dataloaders=dataloaders, verbose=False)

                    accuracy = df_train['avg_acc_val'].mean()
                    # print(f'{new_opt.lr=} {new_opt.momentum=} : {accuracy}')
                    return accuracy

                opt_tuna= dict(storage=f"sqlite:///{os.path.join(data_cache, study_name)}.sqlite3", direction='maximize', load_if_exists=True,study_name=f"{key} = {value}")

                # 3. Create a study object and optimize the objective function.
                study = optuna.create_study(**opt_tuna)
                study.optimize(objective, n_trials=150, n_jobs=1, show_progress_bar=True)
                print(50*'-.')
                print("Best params: ", study.best_params)
                print("Best value: ", study.best_value)
                print("Best Trial: ", study.best_trial)
                print("Trials: ", study.trials)
                print(50*'-.')
                df_scan.loc[i_loc] = {key:value, 'accuracy':study.best_value}
                i_loc += 1
            df_scan.to_json(filename, orient='index', indent=2)
        print(df_scan)
        print(50*'=')

        fig, ax = plt.subplots(figsize=(fig_width, fig_width/phi), subplotpars=subplotpars_scan)
        gp_scan = df_scan[[key, 'accuracy']].groupby([key])
        means = gp_scan.mean()
        errors = gp_scan.std()
        means.plot.bar(yerr=errors, ax=ax, capsize=4, rot=-60, legend=False, color='r', alpha=.5)
        
        ax.set_ylabel('Accuracy')
        ax.set_xlabel(key + ' = ' +label_dicts[key])
        #ax.set_xscale('log')

        ax.set_ylim(0, 1)
        #fig = ax.get_figure()
        # pos = ax.get_position()
        # print(pos)
        plt.show()

Scanning along image_size = image size
i_value 1  / 1 image_size = 224


  0%|          | 0/150 [00:00<?, ?it/s]

[W 2024-05-08 18:19:46,261] Trial 0 failed with parameters: {'rs_min': -0.23412186802758206, 'rs_max': -4.7509653886781775, 'momentum': 0.07620962345802652, 'lr': 6.438218349284374e-05} because of the following error: FileNotFoundError(2, 'No such file or directory').
Traceback (most recent call last):
  File "/opt/homebrew/lib/python3.11/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/3s/q2x8bxzj43g4rdvb2wjt67640000gq/T/ipykernel_98999/36856722.py", line 49, in objective
    model_retrain.load_state_dict(torch.load(model_filename, map_location=torch.device(device)))
                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/torch/serialization.py", line 998, in load
    with _open_file_like(f, 'rb') as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-pac

FileNotFoundError: [Errno 2] No such file or directory: 'cached_data/2024-04-25_focus_resnet101_do_polar=True.pt'